In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from __future__ import annotations

import json
import logging
import pickle
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
'''import shap'''
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
def _build_training_frame(
    users: pd.DataFrame, shifts: pd.DataFrame, events: pd.DataFrame
) -> pd.DataFrame:
    shifts_for_join = shifts.rename(columns={"id": "shift_id"}).copy()
    
    shifts_for_join["reward_per_hour"] = (
        shifts_for_join["reward"] / shifts_for_join["hours"]
    )
        
    merged_events = events.merge(
        shifts_for_join[
            [
                "shift_id",
                "start_at",
                "employer_id",
                "workplace_id",
                "reward_per_hour"   # можно сразу подтянуть
            ]
        ],
        on="shift_id",
        how="inner",
    )
    
    # Prevent leakage from events after shift start.
    merged_events = merged_events[
        merged_events["ts"] <= merged_events["start_at"]
    ].copy()
    
    view_stats = (
        merged_events[
            merged_events["interaction"] == "VIEW"
        ]
        .groupby("user_id")
        .agg(
            user_total_views=("shift_id", "nunique")
        )
        .reset_index()
    )
    
    
    base = grouped.merge(
        shifts_for_join, 
        on="shift_id", 
        how="inner"
    )
    
    # добавляем user_total_views
    base = base.merge(
        view_stats,
        on="user_id",
        how="left"
    )
    
    # если просмотров не было
    base["user_total_views"] = (
        base["user_total_views"]
        .fillna(0)
        .astype(int)
    )
       
    
    return base